# Active Nematics Simulation — Biophysics Laboratory

**Author:** Jorge Arrieta  
**Date:** April 2026  
**Course:** Biophysics

---

## Introduction

This notebook simulates a 2D active nematic system in the dilute limit using the
[Dedalus](https://dedalus-project.org/) spectral PDE framework. Active nematics
describe the collective behaviour of self-propelled elongated particles (bacteria,
cytoskeletal filaments) that generate internal stresses.

### Governing Equations

**Q-tensor evolution (nematodynamic equation):**

$$\frac{D\tilde{\mathbf{Q}}}{Dt} - \tilde{\mathbf{S}} = \frac{1}{Pe} \tilde{\nabla}^2 \mathbf{Q}$$

where $\mathbf{S}$ is the co-rotation tensor that couples the nematic field to the flow:

$$\mathbf{S} = (\lambda \mathbf{E} + \mathbf{\Omega}) \cdot (\mathbf{Q} + \tfrac{\mathbf{I}}{3}) + (\mathbf{Q} + \tfrac{\mathbf{I}}{3}) \cdot (\lambda \mathbf{E} - \mathbf{\Omega}) - 2\lambda (\mathbf{Q} + \tfrac{\mathbf{I}}{3})(\mathbf{Q} : \nabla \mathbf{u})$$

**Stokes flow (streamfunction formulation):**

$$\nabla^4 \psi = -\xi \, \nabla \times (\nabla \cdot \mathbf{Q})$$

with velocity recovered as $u = \partial_y \psi$, $v = -\partial_x \psi$.

### Parameters

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| Péclet number | $Pe$ | Ratio of advection to diffusion |
| Activity | $\xi$ | Strength of active stresses |
| Tumbling parameter | $\lambda$ | Flow alignment of particles |

### Reference

Doostmohammadi, A., Ignés-Mullol, J., Yeomans, J. M. & Sagués, F.  
*Active nematics*, Nature Communications **9**, 3246 (2018)

---
## 1. Installation

Run the following two cells to install Dedalus in Google Colab.  
**Important:** After Cell 1, the runtime will restart automatically. This is normal — just continue with Cell 2.

In [ ]:
# Cell 1 — Install conda (runtime will restart automatically)
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Cell 2 — Install Dedalus (wait 5-10 minutes)
!mamba install -c conda-forge dedalus -y
print("Installation complete!")

In [ ]:
# Cell 3 — Verify installation
!conda run python -c "import dedalus.public as d3; print('Dedalus installed successfully')"

---
## 2. Simulation Code

The cell below writes the simulation script to a file. You can modify the parameters
at the top of the script before running.

**Parameters to experiment with:**
- `xi` — activity: controls the strength of active stresses. Higher values → more turbulent
- `Pe` — Péclet number: controls diffusion. Higher values → sharper structures
- `lambda_n` — tumbling parameter: controls flow alignment
- `Nx, Ny` — grid resolution
- `stop_sim_time` — how long to simulate

In [ ]:
%%writefile active_nematics.py
"""
Active nematics simulation — dilute limit, streamfunction formulation
Periodic boundary conditions in both x and y

Jorge Arrieta, April 2026
Biophysics course
"""
import numpy as np
import dedalus.public as d3

# ======================================================================
# PARAMETERS — Modify these to explore different regimes
# ======================================================================
Lx, Ly   = 1.0, 1.0       # Domain size
Nx, Ny   = 128, 128        # Grid resolution
Pe       = 100              # Peclet number (advection / diffusion)
xi       = 5.0              # Activity parameter
lambda_n = 1.0              # Tumbling parameter
dealias  = 3/2              # Dealiasing factor
stop_sim_time = 0.5         # Total simulation time
timestep = 5e-4             # Time step
save_every = 50             # Save snapshot every N iterations

# ======================================================================
# COORDINATES AND BASES — Fully periodic domain (Fourier in both x, y)
# ======================================================================
coord  = d3.CartesianCoordinates('x', 'y')
dist   = d3.Distributor(coord, dtype=np.complex128)
xbasis = d3.ComplexFourier(coord['x'], size=Nx, bounds=(0, Lx), dealias=dealias)
ybasis = d3.ComplexFourier(coord['y'], size=Ny, bounds=(0, Ly), dealias=dealias)

# ======================================================================
# FIELDS
# ======================================================================
# Streamfunction: u = dy(psi), v = -dx(psi)
# Incompressibility is satisfied automatically
psi     = dist.Field(name='psi',     bases=(xbasis, ybasis))
Qxx     = dist.Field(name='Qxx',     bases=(xbasis, ybasis))
Qxy     = dist.Field(name='Qxy',     bases=(xbasis, ybasis))
tau_psi = dist.Field(name='tau_psi')  # Gauge: fixes mean of psi

# ======================================================================
# OPERATORS
# ======================================================================
dx  = lambda A: d3.Differentiate(A, coord['x'])
dy  = lambda A: d3.Differentiate(A, coord['y'])
lap = lambda A: dx(dx(A)) + dy(dy(A))

# Velocity from streamfunction
u =  dy(psi)
v = -dx(psi)

# ======================================================================
# STRAIN RATE AND VORTICITY
# ======================================================================
Exx   =  dx(u)
Exy   =  (dy(u) + dx(v)) / 2
Eyy   =  dy(v)
Omega =  (dy(u) - dx(v)) / 2

# ======================================================================
# CO-ROTATION TENSOR S (Eq. 2 from Doostmohammadi et al. 2018)
# Qyy = -Qxx by tracelessness
# ======================================================================
# A = lambda*E + Omega
A11 =  lambda_n * Exx
A12 =  lambda_n * Exy + Omega
A21 =  lambda_n * Exy - Omega
A22 = -lambda_n * Exx

# B = Q + I/3
B11 =  Qxx + 1/3
B12 =  Qxy
B21 =  Qxy
B22 = -Qxx + 1/3

# C = lambda*E - Omega
C11 =  lambda_n * Exx
C12 =  lambda_n * Exy - Omega
C21 =  lambda_n * Exy + Omega
C22 = -lambda_n * Exx

# D = Q : nabla(u)
D = Qxx*(dx(u) - dy(v)) + Qxy*(dy(u) + dx(v))

# S = (A.B) + (B.C) - 2*lambda*B*D
S11 = (A11*B11 + A12*B21) + (B11*C11 + B12*C21) - 2*lambda_n*B11*D
S12 = (A11*B12 + A12*B22) + (B11*C12 + B12*C22) - 2*lambda_n*B12*D

# ======================================================================
# ACTIVE FORCE CURL
# Active stress: Pi_active = -zeta * Q (Eq. 9)
# curl(div(Pi_active)) for the biharmonic equation
# ======================================================================
curl_f = -xi * (dx(dx(Qxy)) - 2*dx(dy(Qxx)) - dy(dy(Qxy)))

# ======================================================================
# PROBLEM DEFINITION
# ======================================================================
problem = d3.IVP([psi, Qxx, Qxy, tau_psi], namespace=locals())

# Q-tensor evolution
problem.add_equation("dt(Qxx) + u*dx(Qxx) + v*dy(Qxx) - S11 - (1/Pe)*lap(Qxx) = 0")
problem.add_equation("dt(Qxy) + u*dx(Qxy) + v*dy(Qxy) - S12 - (1/Pe)*lap(Qxy) = 0")

# Biharmonic equation for streamfunction (Stokes + active stress)
problem.add_equation("lap(lap(psi)) + tau_psi - curl_f = 0")

# Gauge condition: mean streamfunction = 0
problem.add_equation("integ(psi) = 0")

# ======================================================================
# INITIAL CONDITIONS
# ======================================================================
Qxx['g'] = 0.001 * np.random.randn(Nx, Ny)
Qxy['g'] = 0.001 * np.random.randn(Nx, Ny)
psi['g'] = 0.0

# ======================================================================
# SOLVER
# ======================================================================
solver = problem.build_solver(d3.SBDF2)
solver.stop_sim_time = stop_sim_time
print(f"Solver built. Running until t = {stop_sim_time}")
print(f"Parameters: Pe={Pe}, xi={xi}, lambda={lambda_n}, Nx={Nx}")

# ======================================================================
# OUTPUT
# ======================================================================
# Remove old snapshots if they exist
import shutil, os
if os.path.exists('snapshots'):
    shutil.rmtree('snapshots')

snapshots = solver.evaluator.add_file_handler('snapshots', iter=save_every, max_writes=1000)
snapshots.add_task(Qxx,      name='Qxx')
snapshots.add_task(Qxy,      name='Qxy')
snapshots.add_task( dy(psi), name='u')
snapshots.add_task(-dx(psi), name='v')
snapshots.add_task(psi,      name='psi')

# ======================================================================
# TIME LOOP
# ======================================================================
while solver.proceed:
    solver.step(timestep)
    if solver.iteration % 100 == 0:
        Qxx_max = np.max(np.abs(Qxx['g'].real))
        S_mean  = np.mean(np.sqrt(Qxx['g'].real**2 + Qxy['g'].real**2))
        print(f"t={solver.sim_time:.4f}  max|Qxx|={Qxx_max:.4e}  <S>={S_mean:.4e}", flush=True)
        if np.isnan(Qxx_max):
            print("NaN detected — stopping")
            break

print("\nSimulation complete!")
print(f"Total iterations: {solver.iteration}")
print(f"Final time: {solver.sim_time:.4f}")

---
## 3. Run the Simulation

Execute the simulation. You should see progress messages every 100 iterations.

In [ ]:
!conda run python -u active_nematics.py 2>&1

---
## 4. Save Results to Google Drive

Mount your Google Drive and copy the results so they are not lost when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
output_dir = '/content/drive/MyDrive/active_nematics'
os.makedirs(output_dir, exist_ok=True)

!cp -r snapshots/ {output_dir}/
print(f"Results saved to {output_dir}")

---
## 5. Load and Inspect Data

Now we load the HDF5 output and explore what was saved.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Find the snapshot files
import glob
files = sorted(glob.glob('snapshots/snapshots_s*.h5'))
print(f"Found {len(files)} snapshot file(s): {files}")

# Load all data
Qxx_all, Qxy_all, u_all, v_all, psi_all, t_all = [], [], [], [], [], []

for fname in files:
    with h5py.File(fname, 'r') as f:
        Qxx_all.append(f['tasks/Qxx'][:])
        Qxy_all.append(f['tasks/Qxy'][:])
        u_all.append(f['tasks/u'][:])
        v_all.append(f['tasks/v'][:])
        psi_all.append(f['tasks/psi'][:])
        t_all.append(f['scales/sim_time'][:])

Qxx = np.concatenate(Qxx_all, axis=0)
Qxy = np.concatenate(Qxy_all, axis=0)
u   = np.concatenate(u_all,   axis=0)
v   = np.concatenate(v_all,   axis=0)
psi = np.concatenate(psi_all, axis=0)
t   = np.concatenate(t_all,   axis=0)

print(f"\nLoaded {len(t)} snapshots")
print(f"Time range: t = {t[0]:.4f} to {t[-1]:.4f}")
print(f"Field shape: {Qxx.shape} (n_frames, Nx, Ny)")
print(f"Data type: {Qxx.dtype}")

---
## 6. Order Parameter Evolution

The scalar order parameter $S = \sqrt{Q_{xx}^2 + Q_{xy}^2}$ measures
how strongly the particles are aligned. $S = 0$ means isotropic (disordered),
$S > 0$ means nematic order.

In [ ]:
# Compute mean order parameter vs time
S_mean = np.array([np.mean(np.sqrt(Qxx[i].real**2 + Qxy[i].real**2)) for i in range(len(t))])
S_max  = np.array([np.max(np.sqrt(Qxx[i].real**2 + Qxy[i].real**2)) for i in range(len(t))])

plt.figure(figsize=(8, 4))
plt.plot(t, S_mean, 'b-', label='Mean S')
plt.plot(t, S_max, 'r-', alpha=0.5, label='Max S')
plt.xlabel('Time')
plt.ylabel('Order parameter S')
plt.title('Growth of nematic order')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Order Parameter Snapshots

Visualise the spatial distribution of nematic order at different times.

In [ ]:
Lx, Ly = 1.0, 1.0
Nx, Ny = Qxx.shape[1], Qxx.shape[2]
x = np.linspace(0, Lx, Nx, endpoint=False)
y = np.linspace(0, Ly, Ny, endpoint=False)
X, Y = np.meshgrid(x, y, indexing='ij')

# Select frames to plot
n_frames = min(4, len(t))
indices = np.linspace(0, len(t)-1, n_frames, dtype=int)

fig, axes = plt.subplots(1, n_frames, figsize=(4*n_frames, 4))
if n_frames == 1:
    axes = [axes]

for ax, idx in zip(axes, indices):
    S = np.sqrt(Qxx[idx].real**2 + Qxy[idx].real**2)
    im = ax.contourf(X, Y, S, cmap='inferno', levels=50)
    ax.set_title(f't = {t[idx]:.3f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, label='S', fraction=0.046)

plt.tight_layout()
plt.show()

---
## 8. Director Field and Defects

The **director** $\mathbf{n}$ is extracted from the Q-tensor:

$$\theta = \frac{1}{2} \arctan\left(\frac{Q_{xy}}{Q_{xx}}\right)$$

$$n_x = \cos\theta, \quad n_y = \sin\theta$$

Topological defects are located where $S \to 0$ (dark regions in the order parameter).
- **+1/2 defects**: comet-shaped, motile
- **-1/2 defects**: trefoil-shaped, nearly stationary

In [ ]:
# Plot director field overlaid on order parameter
frame = -1  # Last frame — change this to explore different times

S = np.sqrt(Qxx[frame].real**2 + Qxy[frame].real**2)
theta = 0.5 * np.arctan2(Qxy[frame].real, Qxx[frame].real)
nx = np.cos(theta)
ny = np.sin(theta)

fig, ax = plt.subplots(figsize=(8, 8))

# Background: order parameter
im = ax.contourf(X, Y, S, cmap='inferno', levels=100)
plt.colorbar(im, ax=ax, label='S', fraction=0.046)

# Overlay: director field (subsampled)
n = 6  # Plot every n-th point
ax.quiver(X[::n,::n], Y[::n,::n], nx[::n,::n], ny[::n,::n],
          color='white', scale=25,
          headwidth=0, headlength=0, headaxislength=0,
          pivot='middle', linewidth=0.5)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Director field and order parameter at t = {t[frame]:.3f}')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## 9. Defect Detection via $|\nabla\theta|$

At a topological defect the director angle $\theta$ is singular, so $|\nabla\theta| \to \infty$.
Peaks in $|\nabla\theta|$ (or equivalently, regions where $S \to 0$) mark defect locations.

In [ ]:
frame = -1

S = np.sqrt(Qxx[frame].real**2 + Qxy[frame].real**2)
theta = 0.5 * np.arctan2(Qxy[frame].real, Qxx[frame].real)
nx = np.cos(theta)
ny = np.sin(theta)

# Gradient of theta (unwrapped to reduce branch cut artifacts)
theta_unwrap = np.unwrap(np.unwrap(theta, axis=0), axis=1)
dtheta_dx = np.gradient(theta_unwrap, x, axis=0)
dtheta_dy = np.gradient(theta_unwrap, y, axis=1)
grad_theta = np.sqrt(dtheta_dx**2 + dtheta_dy**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Order parameter (defects are dark spots where S → 0)
im1 = axes[0].contourf(X, Y, S, cmap='inferno', levels=100)
n = 6
axes[0].quiver(X[::n,::n], Y[::n,::n], nx[::n,::n], ny[::n,::n],
               color='white', scale=25,
               headwidth=0, headlength=0, headaxislength=0,
               pivot='middle', linewidth=0.5)
plt.colorbar(im1, ax=axes[0], label='S', fraction=0.046)
axes[0].set_title('Order parameter S (dark = defects)')

# Right: Gradient of director angle (bright spots = defects)
im2 = axes[1].contourf(X, Y, grad_theta, cmap='hot', levels=100)
plt.colorbar(im2, ax=axes[1], label='|∇θ|', fraction=0.046)
axes[1].set_title('|∇θ| (bright spots = defects)')

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 10. Velocity Field (Vorticity)

The active stresses generate flow. The vorticity $\omega = \partial_x v - \partial_y u$
reveals the characteristic swirling patterns of active turbulence.

In [ ]:
frame = -1

u_frame = u[frame].real
v_frame = v[frame].real

# Vorticity
dvdx = np.gradient(v_frame, x, axis=0)
dudy = np.gradient(u_frame, y, axis=1)
vorticity = dvdx - dudy

# Speed
speed = np.sqrt(u_frame**2 + v_frame**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Vorticity
vmax = np.max(np.abs(vorticity)) * 0.8
im1 = axes[0].contourf(X, Y, vorticity, cmap='RdBu_r', levels=100,
                        vmin=-vmax, vmax=vmax)
plt.colorbar(im1, ax=axes[0], label='ω', fraction=0.046)
axes[0].set_title('Vorticity (red=CCW, blue=CW)')

# Right: Speed with streamlines
im2 = axes[1].contourf(X, Y, speed, cmap='viridis', levels=50)
plt.colorbar(im2, ax=axes[1], label='|u|', fraction=0.046)
# Streamlines
axes[1].streamplot(y, x, u_frame, v_frame, color='white',
                   linewidth=0.5, density=2, arrowsize=0.5)
axes[1].set_title('Speed and streamlines')

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(6, 6))

def animate(i):
    ax.clear()
    S = np.sqrt(Qxx[i].real**2 + Qxy[i].real**2)
    theta = 0.5 * np.arctan2(Qxy[i].real, Qxx[i].real)
    nxi = np.cos(theta)
    nyi = np.sin(theta)

    ax.contourf(X, Y, S, cmap='inferno', levels=50)
    n = 8
    ax.quiver(X[::n,::n], Y[::n,::n], nxi[::n,::n], nyi[::n,::n],
              color='white', scale=25,
              headwidth=0, headlength=0, headaxislength=0,
              pivot='middle', linewidth=0.5)
    ax.set_title(f't = {t[i]:.3f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')

anim = FuncAnimation(fig, animate, frames=len(t), interval=100)
plt.close()
HTML(anim.to_jshtml())

---
## Exercises

Modify the simulation parameters and rerun to explore different regimes.

### Exercise 1 — Effect of activity
Change `xi` from 1 to 10. How does the number of defects change?
Plot the mean order parameter $\langle S \rangle$ vs time for each value of $\xi$.

### Exercise 2 — Effect of diffusion
Change `Pe` from 10 to 1000. What happens to the characteristic size of
the nematic domains? Does the active turbulence become more or less disordered?

### Exercise 3 — Defect counting
At the final time step, count the approximate number of defects visible
in the $S$ field (dark spots). How does this number scale with $\xi$?
Compare with the theoretical prediction that the defect density scales as $\sim \zeta / K$.

### Exercise 4 — Velocity correlations
Compute the spatial velocity-velocity correlation function
$C(r) = \langle \mathbf{u}(\mathbf{x}) \cdot \mathbf{u}(\mathbf{x}+\mathbf{r}) \rangle$.
The active length scale $\ell_a \sim \sqrt{K/\zeta}$ should appear as
the decorrelation length.

### Exercise 5 — Vortex size distribution
Measure the distribution of vortex areas from the vorticity field.
According to Giomi (2015), the distribution should be exponential.
Do your results agree?